# Lab de Limpeza - Camada Silver: Financeiro

In [1]:
import pandas as pd
import numpy as np
import s3fs

In [2]:
storage_options = {
    "key": "minio",
    "secret": "minio123",
    "client_kwargs": {"endpoint_url": "http://localhost:9000"}
}

In [3]:
fs = s3fs.S3FileSystem(**storage_options)
arquivos = fs.glob("bronze/financeiro/**/*.csv")
arquivos

['bronze/financeiro/ingestion_date=2026-04-29T00:30:04/financeiro.csv']

In [4]:
df = pd.concat(
    [
        pd.read_csv(f"s3://{arquivo}", storage_options=storage_options) 
        for arquivo in arquivos
    ],
    ignore_index=True
)
df.head()

,id_financeiro,id_venda,tipo_lancamento,valor_bruto,valor_imposto,valor_liquido,competencia,status,_ingested_at,_source_file
0,F0001614,V002457,AJUSTE,3245.87,389.50,2856.37,202407,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv
1,F0001473,V001995,VENDA,4023.86,482.86,3541.00,202408,CANCELADO,2026-04-29 00:30:04.395742,financeiro.csv
2,F0000317,V003128,AJUSTE,501.30,60.16,441.14,202412,PAGO,2026-04-29 00:30:04.395742,financeiro.csv
3,F0000604,V002963,AJUSTE,1514.17,181.70,1332.47,202412,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv
4,F0000669,V001578,DEVOLUCAO,1362.83,163.54,1199.29,202404,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv


Tipagem

In [5]:
df["_ingested_at"] = pd.to_datetime(df["_ingested_at"], errors="coerce")

cols_float = ["valor_bruto", "valor_imposto", "valor_liquido"]

for col in cols_float:
    df[col] = pd.to_numeric(df[col], errors="coerce")

Padronização de strings

In [6]:
cols_str = ["tipo_lancamento", "status"]

for col in cols_str:
    df[col] = df[col].astype(str).str.strip().str.upper()

Tratamento da competência

In [7]:
df["competencia"].unique()

<ArrowStringArray>
[ '202407',  '202408',  '202412',  '202404',  '202402',  '202411',  '202405',
  '202400',  '202409',  '202406',  '202403',  '202401',  '202410', '2024/01',
 '2024/03', '2024/12', '2024/10', '2024/04', '2024/07', '2024/02', '2024/05',
 '2024/08', '2024/11', '2024/06', '2024/09']
Length: 25, dtype: str

In [8]:
df["competencia_raw"] = df["competencia"]

df["competencia"] = (
    df["competencia"]
    .astype(str)
    .str.replace("/", "", regex=False)
    .str.strip()
)

In [9]:
df["tipo_lancamento"].unique()

<ArrowStringArray>
[       'AJUSTE',         'VENDA',     'DEVOLUCAO',   'BONIFICACAO',
           'VND',         'OUTRO',           'DEV',         'BONIF',
   'DEVOLUÃ§Ã£O',             'V', 'BONIFICAÃ§Ã£O',        'VENDAS',
        'BONIF.',         'DEVOL',             'D']
Length: 15, dtype: str

In [10]:
df["tipo_lancamento_raw"] = df["tipo_lancamento"]

Corrigir encoding

In [11]:
def fix_encoding(text):
    try:
        return text.encode("latin1").decode("utf-8")
    except:
        return text

df["tipo_lancamento"] = (
    df["tipo_lancamento"]
    .astype(str)
    .apply(fix_encoding)
)

Normalizar

In [12]:
df["tipo_lancamento_norm"] = (
    df["tipo_lancamento"]
    .str.strip()
    .str.upper()
    .str.replace(".", "", regex=False)
)

mapeamento semantico

In [13]:
map_tipo = {
    # VENDA
    "VENDA": "VENDA",
    "VENDAS": "VENDA",
    "VND": "VENDA",
    "V": "VENDA",

    # DEVOLUÇÃO
    "DEVOLUCAO": "DEVOLUCAO",
    "DEV": "DEVOLUCAO",
    "DEVOL": "DEVOLUCAO",
    "D": "DEVOLUCAO",
    "DEVOLUÇÃO": "DEVOLUCAO",

    # BONIFICAÇÃO
    "BONIFICACAO": "BONIFICACAO",
    "BONIF": "BONIFICACAO",
    "BONIFICACAO": "BONIFICACAO",
    "BONIFICAÇÃO": "BONIFICACAO",

    # OUTROS
    "AJUSTE": "AJUSTE",
    "OUTRO": "OUTRO",
}

Aplicar e marcar desconhecidos

In [14]:
df["tipo_lancamento_padrao"] = df["tipo_lancamento_norm"].map(map_tipo)

df["tipo_lancamento_desconhecido"] = df["tipo_lancamento_padrao"].isna()

In [15]:
df[df["tipo_lancamento_desconhecido"]]["tipo_lancamento_raw"].value_counts()

Series([], Name: count, dtype: int64)

In [16]:
df["tipo_lancamento_padrao"].unique()

<ArrowStringArray>
['AJUSTE', 'VENDA', 'DEVOLUCAO', 'BONIFICACAO', 'OUTRO']
Length: 5, dtype: str

Status

In [17]:
df["status"].unique()

<ArrowStringArray>
['PENDENTE', 'CANCELADO', 'PAGO', nan]
Length: 4, dtype: str

In [18]:
df["status_raw"] = df["status"]

In [19]:
df["competencia_dt"] = pd.to_datetime(
    df["competencia"],
    format="%Y%m",
    errors="coerce"
)

In [20]:
df["status_norm"] = (
    df["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [21]:
df["status_norm"] = df["status_norm"].replace("NAN", np.nan)

In [22]:
map_status = {
    "PAGO": "PAGO",
    "PENDENTE": "PENDENTE",
    "CANCELADO": "CANCELADO"
}

df["status_padrao"] = df["status_norm"].map(map_status)

In [23]:
df["status_desconhecido"] = df["status_padrao"].isna() & df["status_norm"].notna()
df["status_nulo"] = df["status_norm"].isna()

In [24]:
df["status_padrao"].value_counts(dropna=False)

status_padrao
PAGO         854
PENDENTE     832
CANCELADO    768
NaN           10
Name: count, dtype: int64

In [25]:
df[df["status_desconhecido"]]["status_raw"].value_counts()

Series([], Name: count, dtype: int64)

Detectar inválidos

In [26]:
df["competencia_invalida"] = df["competencia_dt"].isna()

In [27]:
df[df["competencia_invalida"]]["competencia_raw"].value_counts()

competencia_raw
202400    10
Name: count, dtype: int64

In [28]:
df["competencia"] = df["competencia_dt"]

In [29]:
df = df.sort_values("_ingested_at")

df = df.drop_duplicates(
    subset=["id_financeiro"],
    keep="last"
)

In [30]:
df.info()

<class 'pandas.DataFrame'>
Index: 2464 entries, 0 to 2463
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id_financeiro                 2464 non-null   str           
 1   id_venda                      2454 non-null   str           
 2   tipo_lancamento               2464 non-null   str           
 3   valor_bruto                   2464 non-null   float64       
 4   valor_imposto                 2464 non-null   float64       
 5   valor_liquido                 2464 non-null   float64       
 6   competencia                   2454 non-null   datetime64[us]
 7   status                        2454 non-null   str           
 8   _ingested_at                  2464 non-null   datetime64[us]
 9   _source_file                  2464 non-null   str           
 10  competencia_raw               2464 non-null   str           
 11  tipo_lancamento_raw           2464 non-null   

In [31]:
df[["id_financeiro","id_venda","valor_bruto","valor_imposto","valor_liquido","status","_ingested_at","_source_file",]]

,id_financeiro,id_venda,valor_bruto,valor_imposto,valor_liquido,status,_ingested_at,_source_file
0,F0001614,V002457,3245.87,389.50,2856.37,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv
1638,F0001176,V000514,4648.93,557.87,4091.06,CANCELADO,2026-04-29 00:30:04.395742,financeiro.csv
1639,F0002113,V001602,4992.26,599.07,4393.19,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv
1640,F0002154,V000332,2498.57,299.83,2198.74,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv
1641,F0001659,V002551,906.95,108.83,798.12,CANCELADO,2026-04-29 00:30:04.395742,financeiro.csv
...,...,...,...,...,...,...,...,...
823,F0000829,V000707,315.75,37.89,277.86,PAGO,2026-04-29 00:30:04.395742,financeiro.csv
824,F0001882,V002158,737.22,88.47,648.75,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv
825,F0001806,V001692,1187.77,142.53,1045.24,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv
818,F0002429,V003310,2054.06,246.49,1807.57,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv


In [32]:
df['competencia'] = pd.to_datetime(df['competencia'], format='%Y-%m-%d')

In [34]:
df['ano'] = df['competencia'].dt.year
df['mes'] = df['competencia'].dt.month

In [35]:
df.head()

,id_financeiro,id_venda,tipo_lancamento,valor_bruto,valor_imposto,valor_liquido,competencia,status,_ingested_at,_source_file,...,tipo_lancamento_desconhecido,status_raw,competencia_dt,status_norm,status_padrao,status_desconhecido,status_nulo,competencia_invalida,ano,mes
0,F0001614,V002457,AJUSTE,3245.87,389.50,2856.37,2024-07-01,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv,...,False,PENDENTE,2024-07-01,PENDENTE,PENDENTE,False,False,False,2024.0,7.0
1638,F0001176,V000514,AJUSTE,4648.93,557.87,4091.06,2024-01-01,CANCELADO,2026-04-29 00:30:04.395742,financeiro.csv,...,False,CANCELADO,2024-01-01,CANCELADO,CANCELADO,False,False,False,2024.0,1.0
1639,F0002113,V001602,BONIFICACAO,4992.26,599.07,4393.19,2024-07-01,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv,...,False,PENDENTE,2024-07-01,PENDENTE,PENDENTE,False,False,False,2024.0,7.0
1640,F0002154,V000332,DEVOLUCAO,2498.57,299.83,2198.74,2024-01-01,PENDENTE,2026-04-29 00:30:04.395742,financeiro.csv,...,False,PENDENTE,2024-01-01,PENDENTE,PENDENTE,False,False,False,2024.0,1.0
1641,F0001659,V002551,AJUSTE,906.95,108.83,798.12,2024-04-01,CANCELADO,2026-04-29 00:30:04.395742,financeiro.csv,...,False,CANCELADO,2024-04-01,CANCELADO,CANCELADO,False,False,False,2024.0,4.0


In [36]:
df.columns

Index(['id_financeiro', 'id_venda', 'tipo_lancamento', 'valor_bruto',
       'valor_imposto', 'valor_liquido', 'competencia', 'status',
       '_ingested_at', '_source_file', 'competencia_raw',
       'tipo_lancamento_raw', 'tipo_lancamento_norm', 'tipo_lancamento_padrao',
       'tipo_lancamento_desconhecido', 'status_raw', 'competencia_dt',
       'status_norm', 'status_padrao', 'status_desconhecido', 'status_nulo',
       'competencia_invalida', 'ano', 'mes'],
      dtype='str')

In [38]:
df_silver = df[[
    "id_financeiro",
    "id_venda",
    "tipo_lancamento_padrao",
    "competencia",
    "status_padrao",
    "ano",
    "mes"
]]

In [45]:
df_silver

,id_financeiro,id_venda,tipo_lancamento_padrao,competencia,status_padrao,ano,mes
0,F0001614,V002457,AJUSTE,2024-07-01,PENDENTE,2024.0,7.0
1638,F0001176,V000514,AJUSTE,2024-01-01,CANCELADO,2024.0,1.0
1639,F0002113,V001602,BONIFICACAO,2024-07-01,PENDENTE,2024.0,7.0
1640,F0002154,V000332,DEVOLUCAO,2024-01-01,PENDENTE,2024.0,1.0
1641,F0001659,V002551,AJUSTE,2024-04-01,CANCELADO,2024.0,4.0
...,...,...,...,...,...,...,...
823,F0000829,V000707,AJUSTE,2024-12-01,PAGO,2024.0,12.0
824,F0001882,V002158,DEVOLUCAO,2024-12-01,PENDENTE,2024.0,12.0
825,F0001806,V001692,DEVOLUCAO,2024-07-01,PENDENTE,2024.0,7.0
818,F0002429,V003310,AJUSTE,2024-01-01,PENDENTE,2024.0,1.0


In [44]:
df_silver = df_silver.dropna(subset=["ano"])

In [46]:
try:
    df_silver.to_parquet(
        path="s3://silver/financeiro/",
        partition_cols=["ano", "mes"],
        storage_options=storage_options,
        index=False,
        compression="snappy"
    )
    print("Dados salvos com sucesso na camada Silver (MinIO)!")
except Exception as e:
    print(f"Erro ao salvar na camada Silver: {e}")

Dados salvos com sucesso na camada Silver (MinIO)!
